# LOSO + False-Negative (Standalone)
Jalankan TANPA pipeline berat. Isi `DATA_DIR` (CSV) & `MODEL_DIR` (best_model) di sel Setup.

- **LOSO** butuh CSV saja (~2-3 jam untuk 4 sumber; kurangi `leave_out_sources`/`train_subset`/`epochs` bila mepet).
- **FN** butuh best_model + Internet (stream held-out); ~10 menit.

## Setup

In [ ]:
# ============ SETUP (standalone: TANPA pipeline berat) ============
# Prasyarat di Kaggle:
#  1) Upload train.csv, val.csv, test.csv sebagai Kaggle Dataset -> set DATA_DIR.
#  2) (untuk FN) Upload folder best_model (hasil unzip best_model_biner.zip) -> set MODEL_DIR.
#  3) Aktifkan GPU + Internet.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "transformers==4.46.3", "datasets", "scikit-learn"])
import os, gc, re, time, dataclasses, inspect, numpy as np, pandas as pd, torch
import torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONFIG = {"labels": ["safe", "prompt_injection"], "model_name": "microsoft/deberta-v3-base",
          "max_length": 192, "batch_size": 8, "grad_accum": 2, "weight_decay": 0.01, "warmup_ratio": 0.06,
          "heldout_sources": {"safe": {"id": "openai/gsm8k", "config": "main", "text_col": "question"},
                              "prompt_injection": {"id": "Lakera/gandalf_ignore_instructions", "text_col": "text"}},
          "heldout_per_class": 2000}
LABEL2ID = {l: i for i, l in enumerate(CONFIG["labels"])}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

# >>> SESUAIKAN PATH <<<
DATA_DIR  = "/kaggle/input/skripsi-biner-csv"             # folder berisi train.csv/val.csv/test.csv
MODEL_DIR = "/kaggle/input/best-model-biner/best_model"   # folder best_model (untuk FN)
OUTPUT_DIR = "/kaggle/working"
CKPT_DIR = "/kaggle/working/ckpt"; os.makedirs(CKPT_DIR, exist_ok=True)

_TA_FIELDS = {f.name for f in dataclasses.fields(TrainingArguments)}
_TR_PARAMS = set(inspect.signature(Trainer.__init__).parameters)
def _ta_kwargs(**kw):
    if "eval_strategy" not in _TA_FIELDS and "evaluation_strategy" in _TA_FIELDS and "eval_strategy" in kw:
        kw["evaluation_strategy"] = kw.pop("eval_strategy")
    for k in [k for k in list(kw) if k not in _TA_FIELDS]:
        kw.pop(k)
    return kw

def preprocess(t):
    import unicodedata
    t = str(t)
    for z in ["​", "‌", "‍", "﻿", "⁠"]:
        t = t.replace(z, "")
    t = unicodedata.normalize("NFKC", t)
    return re.sub(r"\s+", " ", t).strip().lower()

print("Setup OK. DEVICE =", DEVICE)


## Bagian A - LOSO

In [ ]:
# ============ Bagian A — LOSO (Leave-One-Source-Out) ============
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

LOSO = {"run": True, "model_name": CONFIG["model_name"],
        "leave_out_sources": ["jayavibhav/prompt-injection",
                              "imoxto/prompt_injection_cleaned_dataset-v2",
                              "reshabhs/SPML_Chatbot_Prompt_Injection",
                              "xTRam1/safe-guard-prompt-injection"],
        "train_subset": 60_000, "epochs": 2, "learning_rate": 2e-5, "test_cap": 5000}

if LOSO["run"]:
    full = pd.concat([pd.read_csv(os.path.join(DATA_DIR, f"{s}.csv")) for s in ["train", "val", "test"]], ignore_index=True)
    sbf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    rows = []
    for S in LOSO["leave_out_sources"]:
        tr_pool = full[full["source"] != S]
        te = full[full["source"] == S].copy()
        if len(te) == 0:
            continue
        if len(te) > LOSO["test_cap"]:
            te = te.sample(LOSO["test_cap"], random_state=SEED)
        tr = train_test_split(tr_pool, train_size=LOSO["train_subset"], stratify=tr_pool["label_id"], random_state=SEED)[0] \
            if (LOSO["train_subset"] and len(tr_pool) > LOSO["train_subset"]) else tr_pool
        y_te, pos = te["label_id"].values, LABEL2ID["prompt_injection"]
        print(f"\n== leave-out {S} (latih {len(tr)}, uji {len(te)}) ==")
        bl = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000, sublinear_tf=True)),
                       ("clf", LogisticRegression(max_iter=2000, n_jobs=-1, C=5.0))])
        bl.fit(tr["text_clean"], tr["label_id"])
        rec_t = recall_score(y_te, bl.predict(te["text_clean"]), pos_label=pos)
        tk = AutoTokenizer.from_pretrained(LOSO["model_name"])
        def _p(df):
            ds = Dataset.from_pandas(df[["text_clean", "label_id"]].rename(columns={"text_clean": "text", "label_id": "labels"}), preserve_index=False)
            return ds.map(lambda b: tk(b["text"], truncation=True, max_length=CONFIG["max_length"]), batched=True).remove_columns(["text"])
        mdl = AutoModelForSequenceClassification.from_pretrained(LOSO["model_name"], num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)
        args = TrainingArguments(**_ta_kwargs(output_dir=os.path.join(CKPT_DIR, "loso_" + S.split("/")[-1]),
            learning_rate=LOSO["learning_rate"], per_device_train_batch_size=CONFIG["batch_size"],
            per_device_eval_batch_size=CONFIG["batch_size"], gradient_accumulation_steps=CONFIG["grad_accum"],
            num_train_epochs=LOSO["epochs"], weight_decay=CONFIG["weight_decay"], warmup_ratio=CONFIG["warmup_ratio"],
            save_strategy="no", fp16=False, bf16=sbf16, report_to="none", seed=SEED, logging_steps=200, max_grad_norm=1.0))
        trn = Trainer(model=mdl, args=args, train_dataset=_p(tr), data_collator=DataCollatorWithPadding(tokenizer=tk),
                      **({"processing_class": tk} if "processing_class" in _TR_PARAMS else {"tokenizer": tk}))
        trn.train()
        pred = np.argmax(trn.predict(_p(te)).predictions, axis=-1)
        rec_d = recall_score(y_te, pred, pos_label=pos)
        rows.append({"Sumber": S.split("/")[-1], "n uji": len(te), "Recall TF-IDF": round(rec_t, 4),
                     "Recall DeBERTa": round(rec_d, 4), "Selisih": round(rec_d - rec_t, 4)})
        print(rows[-1])
        del mdl, trn; gc.collect(); torch.cuda.empty_cache()
    loso_df = pd.DataFrame(rows); loso_df.to_csv(os.path.join(OUTPUT_DIR, "loso_results.csv"), index=False)
    print("\n=== LOSO ==="); print(loso_df.to_string(index=False))


## Bagian B - Muat model & held-out

In [ ]:
# ============ Bagian B — Muat best_model + bangun held-out (untuk FN) ============
from datasets import load_dataset
from sklearn.metrics import classification_report
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE).eval()

def collect(sp, n, cls):
    ds = load_dataset(sp["id"], sp.get("config"), split="train", streaming=True)
    txt = []
    for ex in ds:
        t = ex.get(sp["text_col"])
        if isinstance(t, str) and len(t.strip()) >= 3:
            txt.append(t.strip()[:2000])
            if len(txt) >= n:
                break
    return pd.DataFrame({"text": txt, "label": cls})

frames = [collect(sp, CONFIG["heldout_per_class"], cls) for cls, sp in CONFIG["heldout_sources"].items()]
df_ho = pd.concat(frames, ignore_index=True)
df_ho["text_clean"] = df_ho["text"].map(preprocess)
df_ho = df_ho[df_ho["text_clean"].str.len() >= 3].drop_duplicates("text_clean").reset_index(drop=True)
print("held-out:", df_ho["label"].value_counts().to_dict())

@torch.no_grad()
def predict(texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = tokenizer(list(texts[i:i+bs]), truncation=True, max_length=CONFIG["max_length"], padding=True, return_tensors="pt").to(DEVICE)
        out.extend(np.argmax(F.softmax(model(**enc).logits, dim=-1).cpu().numpy(), axis=-1).tolist())
    return out

y_pred_ho = predict(df_ho["text_clean"].tolist())
print(classification_report(df_ho["label"].map(LABEL2ID), y_pred_ho, target_names=CONFIG["labels"], digits=4))


## Bagian C - Analisis False Negative

In [ ]:
# ============ Bagian C — Analisis False Negative Held-out ============
df_fn = df_ho.copy(); df_fn["pred"] = [ID2LABEL[i] for i in y_pred_ho]
fn = df_fn[(df_fn["label"] == "prompt_injection") & (df_fn["pred"] == "safe")].copy()
print(f"False negative: {len(fn)} dari {(df_fn['label'] == 'prompt_injection').sum()} serangan")

@torch.no_grad()
def _conf(texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = tokenizer(list(texts[i:i+bs]), truncation=True, max_length=CONFIG["max_length"], padding=True, return_tensors="pt").to(DEVICE)
        out.extend(F.softmax(model(**enc).logits, dim=-1).cpu().numpy()[:, LABEL2ID["prompt_injection"]].tolist())
    return out

fn["skor_injeksi"] = _conf(fn["text_clean"].tolist()) if len(fn) else []
def _cat(t):
    tl = t.lower(); tags = []
    if len(tl.split()) <= 4: tags.append("sangat_pendek")
    if re.search(r"[^\x00-\x7f]", t): tags.append("non-ascii/multibahasa")
    if any(k in tl for k in ["you are", "act as", "pretend", "roleplay", "role-play", "persona", "you're now"]): tags.append("role-play")
    if any(k in tl for k in ["please", "kindly", "could you", "would you", "tolong", "mohon"]): tags.append("sopan/benign-looking")
    if re.search(r"[_\-=*/\\|]{3,}", t) or re.search(r"\b\w{20,}\b", t): tags.append("obfuscated")
    if "```" in t or "def " in tl or "{" in t or "</" in t: tags.append("code/markup")
    return ",".join(tags) if tags else "lainnya"
fn["pola"] = fn["text_clean"].map(_cat)
print("\n=== Sebaran pola FN ===")
for k, v in fn["pola"].str.split(",").explode().value_counts().items():
    print(f"  {k:24s} {v}")
fn[["text_clean", "skor_injeksi", "pola"]].sort_values("skor_injeksi").to_csv(os.path.join(OUTPUT_DIR, "false_negative_heldout.csv"), index=False)
print("\nDisimpan false_negative_heldout.csv")
with pd.option_context("display.max_colwidth", 90):
    display(fn.sort_values("skor_injeksi")[["text_clean", "skor_injeksi", "pola"]].head(15))
